# COMP 3200 — Assignment 6
## The Trial of Reflection


In [35]:
import numpy as np

# 4 sensings, 3 binary tells each
tells = np.array([[1, 0, 1], # foot shift, no guard drop, exhale
[0, 1, 1], # no shift, guard drop, exhale
[0, 0, 1], # only exhale
[1, 1, 1]]) # all three (the bluff)

# Ground truth: 1 = strike imminent, 0 = they will hold
strike = np.array([[1, 1, 0, 0]]).T # column vector, shape (4, 1)


## Part 1 — Single Layer

The single layer should fail because the pattern cannot be represented with one weighted sum. The same input features need different results depending on the combination of tells, so the error should decrease but not reach zero.


In [36]:
def single_layer_train(tells, strike, alpha, epochs, seed, verbose = False, debug = False):
    """
    This function trains with a single layer, no hidden layers.
    """
    np.random.seed(seed)
    weights = 2 * np.random.random(3) - 1
    mse_hist = []
    total_error_hist = []
    for epoch in range(epochs):
        total_error = 0
        for i in range(len(tells)):
            pred = tells[i].dot(weights)
            delta = pred - strike[i]
            weights -= alpha * delta * tells[i]
            total_error += (pred - strike[i]) ** 2
            mse = total_error / len(tells)
            mse_hist.append(mse)
            total_error_hist.append(total_error)
        if epoch % 10 == 0:
            if verbose:
                print(f"epoch {epoch + 1:>2}: MSE = {mse:.4f}")
            if debug:
                print(f"epoch {epoch + 1:>2}: total_error = {total_error:.4f}")
    if debug:
        print(f"Final Weight: {weights}")

    return weights, mse_hist, total_error_hist

alpha = 0.1
epochs = 60
seed = 1

weights, mse_hist, total_error_hist = single_layer_train(
    tells, strike.ravel(), alpha, epochs, seed, verbose=True, debug=True
)

print("\nFinal weights:", weights)


epoch  1: MSE = 1.7289
epoch  1: total_error = 6.9157
epoch 11: MSE = 0.3859
epoch 11: total_error = 1.5437
epoch 21: MSE = 0.3325
epoch 21: total_error = 1.3300
epoch 31: MSE = 0.3172
epoch 31: total_error = 1.2690
epoch 41: MSE = 0.3122
epoch 41: total_error = 1.2486
epoch 51: MSE = 0.3102
epoch 51: total_error = 1.2409
Final Weight: [ 0.00937726 -0.04602622  0.43220539]

Final weights: [ 0.00937726 -0.04602622  0.43220539]


The error gets smaller, but the model does not reach zero error. That failure is actually what we want to see here because it shows why one layer is not enough for this pattern.

## Part 2 — Forward Pass

Now we add a hidden layer with 4 units. This gives the network another layer that can combine the three tells into different patterns before producing the final prediction.


In [37]:
def relu(x):
    zeroed = x * (x > 0)
    return zeroed

#results experimentally verified very sloppily by using np.ones for the weights
# seems to work

def forward(layer_0, weights_0_1, weights_1_2):
    # first layer is equal to relu of the product of the weight matrix and the input.
    # So for e.g. our 3x4 weight matrix and our 1x3 input vector
    # here, we're multiplying a 1x3 vector by a 3x4 matrix, and relu of the dot products 
    # of each column with the vector are our 1x4 outputs.
    layer_1 = relu(layer_0@weights_0_1)
    #print(layer_1.shape)

    # second layer is just product of the input and the weight matrix, no relu;
    # so e.g. our 1x4 input vector and our 4x1 weight matrix will return their 
    # dot product, a scalar.
    layer_2 = layer_1@weights_1_2
    #print(layer_2.shape)

    # output both
    return layer_1, layer_2


np.random.seed(1)
hidden_size = 4
weights_0_1 = 2 * np.random.random((3, hidden_size)) - 1
weights_1_2 = 2 * np.random.random((hidden_size, 1)) - 1

print("weights_0_1 shape:", weights_0_1.shape)
print("weights_1_2 shape:", weights_1_2.shape)

for i in range(len(tells)):
    layer_1, layer_2 = forward(tells[i:i+1], weights_0_1, weights_1_2)
    print(f"Sensing {i}: layer 1 = {layer_1}, layer 2 = {layer_2}")


weights_0_1 shape: (3, 4)
weights_1_2 shape: (4, 1)
Sensing 0: layer 1 = [[-0.          0.51828245 -0.         -0.        ]], layer 2 = [[0.39194327]]
Sensing 1: layer 1 = [[-0.         -0.         -0.          0.06156045]], layer 2 = [[0.02098811]]
Sensing 2: layer 1 = [[-0.          0.07763347 -0.          0.370439  ]], layer 2 = [[0.18500476]]
Sensing 3: layer 1 = [[-0. -0. -0. -0.]], layer 2 = [[0.]]


The shapes are `(1, 3) @ (3, 4) → (1, 4)` and `(1, 4) @ (4, 1) → (1, 1)`. The first multiplication creates the 4 hidden values, and the second turns those into one prediction.


## Part 3 — One Backpropagation Step

This does one forward pass, calculates the error, and updates the weights. The important part is that the error is passed backward so both sets of weights can be adjusted.


In [38]:
def relu2deriv(y):
    return y > 0


# important note: the layer_2 this returns is the value before
# the update step, pre-modification of weights. The layer_2_error
# this returns, ditto

# layer_0 and target will be row vectors, specifically 1x3 and 1x1 here

def one_step(layer_0, target, weights_0_1, weights_1_2, alpha):

    # forward pass
    # also row vectors, here 1x4 and 1x1
    layer_1, layer_2 = forward(layer_0, weights_0_1, weights_1_2)

    # layer 2 delta is easy: it's the difference between pred and true
    layer_2_delta = layer_2 - target

    # squaring scalars is also easy
    layer_2_error = layer_2_delta**2


    # this one's muddlier and I really, really need to work through
    # reverse chain rule again some time to make sure I have this down.

    layer_1_delta = layer_2_delta @ weights_1_2.T * relu2deriv(layer_1)

    

    

    # now it's backprop time!

    # the weight update is alpha times the outer product of our 3 inputs and
    # our 4 deltas: each column is the relevant delta x the input set,
    # so each column of the weights (corresponding to the operations on a set
    # of inputs to get one of the outputs) will be modified by the inputs x
    # the delta relating to that output
    
    # bleh, np.outer() does of course work on both row and column vectors in 
    # any configuration...
    updated_weights_0_1 = weights_0_1 - alpha * np.outer(layer_0, layer_1_delta)

    # same logic, but more trivial because it's got more 1s
    updated_weights_1_2 = weights_1_2 - alpha * np.outer(layer_1, layer_2_delta)

    
    return updated_weights_0_1, updated_weights_1_2, layer_2, layer_2_error


np.random.seed(1)
weights_0_1 = 2 * np.random.random((3, 4)) - 1
weights_1_2 = 2 * np.random.random((4, 1)) - 1

layer_1_before, layer_2_before = forward(tells[0:1], weights_0_1, weights_1_2)
error_before = (layer_2_before - strike[0:1]) ** 2

new_weights_0_1, new_weights_1_2, old_layer_2, old_error = one_step(
    tells[0:1], strike[0:1], weights_0_1, weights_1_2, alpha=0.2
)
_, layer_2_after = forward(tells[0:1], new_weights_0_1, new_weights_1_2)
error_after = (layer_2_after - strike[0:1]) ** 2

print("Before:", layer_2_before, error_before)
print("After:", layer_2_after, error_after)
print("Weight shapes:", new_weights_0_1.shape, new_weights_1_2.shape)


Before: [[0.39194327]] [[0.36973299]]
After: [[0.57530017]] [[0.18036995]]
Weight shapes: (3, 4) (4, 1)


For the selected weight, the update is `new = old - alpha × layer_1 × layer_2_delta`. The transpose is needed so the dimensions work, and the ReLU derivative makes sure inactive hidden units do not get updated from that path.

## Part 4 — Full Training

Now the same update is repeated for 60 epochs. Instead of looking at one update, we can see whether the network eventually learns the whole pattern.


In [39]:
def train(tells, strike, alpha, epochs, hidden_size, seed, verbose = False):
    # Set random seed so results can be reproduced
    np.random.seed(seed)

    # Initialize weights randomly between -1 and 1
    weights_0_1 = 2 * np.random.random((3, hidden_size)) - 1 # layer 0 to layer 1
    weights_1_2 = 2 * np.random.random((hidden_size, 1)) - 1 # layer 1 to layer 2

    # Track total error for each epoch
    error_history = []

    # Loop through the dataset for the given number of epochs
    for epoch in range(epochs):
        total_epoch_error = 0.0

        # Process each sensing sample one at a time (Stochastic GD)
        for i in range(len(tells)):
            layer_0 = tells[i] # current input sensing vector
            target = strike[i] # current ground truth target

            # Perform one forward pass and update weights using backprop
            weights_0_1, weights_1_2, layer_2, layer_2_error = one_step(
                layer_0, target, weights_0_1, weights_1_2, alpha
            )

            # Add current sample's error to the total epoch error
            total_epoch_error += float(layer_2_error[0])

        # Save this epoch's total error
        error_history.append(total_epoch_error)

        # Print total squared error every 10 epochs
        # +1 so it doesn't print right away
        if (epoch + 1) % 10 == 0:
            if verbose:
                print(f"Epoch {epoch + 1:2d} | Total Error: {total_epoch_error:.6f}")

    # Return updated weights and error log
    return weights_0_1, weights_1_2, error_history

alpha = 0.2
epochs = 60
hidden_size = 4
seed = 1

weights_0_1, weights_1_2, error_history = train(
    tells, strike, alpha, epochs, hidden_size, seed, verbose=True
)

print("\nFinal predictions:")
for i in range(len(tells)):
    _, pred = forward(tells[i], weights_0_1, weights_1_2)
    print(f"Sensing {i}: {float(pred[0]):.4f} | Goal: {strike[i][0]}")


Epoch 10 | Total Error: 0.634231
Epoch 20 | Total Error: 0.358384
Epoch 30 | Total Error: 0.083018
Epoch 40 | Total Error: 0.006467
Epoch 50 | Total Error: 0.000329
Epoch 60 | Total Error: 0.000015

Final predictions:
Sensing 0: 1.0000 | Goal: 1
Sensing 1: 0.9986 | Goal: 1
Sensing 2: 0.0026 | Goal: 0
Sensing 3: 0.0000 | Goal: 0


The predictions end up on the correct side of 0.5, so the network has learned to separate the strike cases from the hold cases.


In [40]:
print("weights_0_1:")
print(weights_0_1.round(2))
print("\nweights_1_2:")
print(weights_1_2.round(2))


weights_0_1:
[[-0.17  0.91 -1.   -0.9 ]
 [-0.71 -0.93 -0.63  0.9 ]
 [-0.21 -0.03 -0.16  0.  ]]

weights_1_2:
[[-0.59]
 [ 1.14]
 [-0.95]
 [ 1.11]]


The network discovered an internal vocabulary. Looking at the learned weights, the hidden units respond to different combinations of the tells. For example, one unit responds to the foot shift without the guard drop, another to the guard drop without the foot shift, and another helps suppress the bluff case. One unit stays inactive.


### Hidden-Size Sweep


In [41]:
sizes = [1, 2, 4, 8, 16]
seeds = [1, 2, 3]

for h_size in sizes:
    for s in seeds:
        _, _, errs = train(tells, strike, alpha=0.2, epochs=60, hidden_size=h_size, seed=s)
        print(f"Hidden Size: {h_size:2d} | Seed: {s} | Final Total Error: {errs[-1]:.6f}")


Hidden Size:  1 | Seed: 1 | Final Total Error: 2.000000
Hidden Size:  1 | Seed: 2 | Final Total Error: 2.000095
Hidden Size:  1 | Seed: 3 | Final Total Error: 2.000004
Hidden Size:  2 | Seed: 1 | Final Total Error: 2.000000
Hidden Size:  2 | Seed: 2 | Final Total Error: 2.000000
Hidden Size:  2 | Seed: 3 | Final Total Error: 0.066403
Hidden Size:  4 | Seed: 1 | Final Total Error: 0.000015
Hidden Size:  4 | Seed: 2 | Final Total Error: 1.000000
Hidden Size:  4 | Seed: 3 | Final Total Error: 1.000000
Hidden Size:  8 | Seed: 1 | Final Total Error: 0.000000
Hidden Size:  8 | Seed: 2 | Final Total Error: 0.033719
Hidden Size:  8 | Seed: 3 | Final Total Error: 0.000000
Hidden Size: 16 | Seed: 1 | Final Total Error: 0.000000
Hidden Size: 16 | Seed: 2 | Final Total Error: 0.000000
Hidden Size: 16 | Seed: 3 | Final Total Error: 0.000000


Size 1 fails for all three seeds. Size 2 only works for one seed. Size 4 works for seed 1 but not seeds 2 and 3. Sizes 8 and 16 work for all three tested seeds. This shows that the random starting weights can affect whether a smaller network finds a useful solution.

## Part 5 — Unit Tests

Run the supplied test file. These tests check the main pieces of the network and make sure the final training still converges.

In [42]:
# Run the supplied tests
%run test_trial.py


 PASS: test_backprop_step
 PASS: test_determinism
 PASS: test_forward_shapes
 PASS: test_full_train_converge
 PASS: test_relu
 PASS: test_single_layer_failure


## Final Reflection

The hidden layer makes the Trial pattern learnable. Backpropagation adjusts the weights so the network can separate the strike and hold cases. The main thing I saw was that the single layer could not represent the pattern, but the hidden layer gave the network enough structure to learn it.
